## Welcome to the Second Lab - Exercise: Advanced Agentic Design Patterns

This notebook extends the previous lab by adding the **Reflection Pattern** to improve response quality.

### Patterns used in the original lab:
1. **Multi-Model Comparison Pattern** - Comparing multiple models
2. **Judge/Evaluator Pattern** - Evaluation by a judge model

### New pattern added:
3. **Reflection Pattern** - Self-improvement of responses

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">New Pattern: Reflection</h2>
            <span style="color:#ff7800;">The Reflection Pattern allows a model to critique and improve its own response. This is particularly useful for complex tasks requiring nuance and precision.</span>
        </td>
    </tr>
</table>

In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

## Step 1: Generate Initial Question (Multi-Model Pattern)

In [ ]:
# Generate a challenging question for the models to answer

request = "Please come up with a challenging ethical dilemma that requires careful moral reasoning and consideration of multiple perspectives. "
request += "The dilemma should involve conflicting values and have no clear-cut answer. Answer only with the dilemma, no explanation."

messages = [{"role": "user", "content": request}]

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)

question = response.choices[0].message.content
print("Generated Question:")
print(question)

## Step 2: Get Initial Responses from Multiple Models

In [ ]:
def get_initial_response(client, model_name, question, is_anthropic=False):
    """Get initial response from a model"""
    messages = [{"role": "user", "content": question}]
    
    if is_anthropic:
        response = client.messages.create(
            model=model_name, 
            messages=messages, 
            max_tokens=1000
        )
        return response.content[0].text
    else:
        response = client.chat.completions.create(
            model=model_name, 
            messages=messages
        )
        return response.choices[0].message.content

In [ ]:
# Configure clients
openai_client = OpenAI()
claude_client = Anthropic() if anthropic_api_key else None
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if google_api_key else None
deepseek_client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1") if deepseek_api_key else None
groq_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1") if groq_api_key else None

In [ ]:
# Collect initial responses
initial_responses = {}
competitors = []

models = [
    ("gpt-4o-mini", openai_client, False),
    ("claude-3-7-sonnet-latest", claude_client, True),
    ("gemini-2.0-flash", gemini_client, False),
    ("deepseek-chat", deepseek_client, False),
    ("llama-3.3-70b-versatile", groq_client, False),
]

print("\n=== INITIAL RESPONSES ===\n")

for model_name, client, is_anthropic in models:
    if client:
        try:
            response = get_initial_response(client, model_name, question, is_anthropic)
            initial_responses[model_name] = response
            competitors.append(model_name)
            
            print(f"**{model_name}:**")
            display(Markdown(response))
            print("\n" + "="*50 + "\n")
        except Exception as e:
            print(f"Error with {model_name}: {e}")

## Step 3: NEW PATTERN - Reflection Pattern

In [ ]:
def apply_reflection_pattern(client, model_name, original_question, initial_response, is_anthropic=False):
    """Apply the Reflection Pattern to improve a response"""
    
    reflection_prompt = f"""
You previously received this question:
{original_question}

Here was your initial response:
{initial_response}

Now, as a critical expert, analyze your own response:
1. What are the strengths of this response?
2. What important perspectives are missing?
3. Are there any biases or blind spots in the analysis?
4. How could you improve this response?

After this self-critique, provide an IMPROVED response that takes into account your observations.

Response format:
## Self-Critique
[Your critical analysis of the initial response]

## Improved Response
[Your revised and improved response]
"""
    
    messages = [{"role": "user", "content": reflection_prompt}]
    
    if is_anthropic:
        response = client.messages.create(
            model=model_name, 
            messages=messages, 
            max_tokens=1500
        )
        return response.content[0].text
    else:
        response = client.chat.completions.create(
            model=model_name, 
            messages=messages
        )
        return response.choices[0].message.content

In [ ]:
# Apply Reflection Pattern
reflected_responses = {}

print("\n=== RESPONSES AFTER REFLECTION ===\n")

for model_name, client, is_anthropic in models:
    if client and model_name in initial_responses:
        try:
            reflected = apply_reflection_pattern(
                client, model_name, question, 
                initial_responses[model_name], is_anthropic
            )
            reflected_responses[model_name] = reflected
            
            print(f"**{model_name} - After Reflection:**")
            display(Markdown(reflected))
            print("\n" + "="*50 + "\n")
        except Exception as e:
            print(f"Error with reflection for {model_name}: {e}")

## Step 4: Comparative Evaluation (Extended Judge Pattern)

In [ ]:
def create_comparative_evaluation(question, initial_responses, reflected_responses):
    """Create a comparative evaluation of responses before/after reflection"""
    
    evaluation_prompt = f"""
You are evaluating the effectiveness of the "Reflection Pattern" on the following question:
{question}

For each model, you have:
1. An initial response
2. A response after self-reflection

Analyze and compare:
- Depth of analysis
- Consideration of multiple perspectives
- Nuance and sophistication of reasoning
- Improvement brought by reflection

MODELS TO EVALUATE:
"""
    
    for model_name in initial_responses:
        if model_name in reflected_responses:
            evaluation_prompt += f"""
## {model_name}

### Initial response:
{initial_responses[model_name][:500]}...

### Response after reflection:
{reflected_responses[model_name][:800]}...

"""
    
    evaluation_prompt += """
Respond with structured JSON:
{
    "general_analysis": "Your analysis of the Reflection Pattern's effectiveness",
    "initial_ranking": ["best initially ranked model", "second", "third"],
    "post_reflection_ranking": ["best ranked model after reflection", "second", "third"],
    "most_improved": "Which model improved the most",
    "insights": "Insights about the usefulness of the Reflection Pattern"
}
"""
    
    return evaluation_prompt

In [ ]:
# Final evaluation
if initial_responses and reflected_responses:
    evaluation_prompt = create_comparative_evaluation(question, initial_responses, reflected_responses)
    
    judge_messages = [{"role": "user", "content": evaluation_prompt}]
    
    try:
        judge_response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=judge_messages,
        )
        
        evaluation_result = judge_response.choices[0].message.content
        print("\n=== FINAL EVALUATION ===\n")
        print(evaluation_result)
        
        # Try to parse JSON for structured display
        try:
            eval_json = json.loads(evaluation_result)
            print("\n=== STRUCTURED RESULTS ===\n")
            for key, value in eval_json.items():
                print(f"{key.replace('_', ' ').title()}: {value}")
        except:
            print("Could not parse JSON, raw output shown above")
            
    except Exception as e:
        print(f"Error during final evaluation: {e}")

## Simple Before/After Comparison

In [ ]:
# Display side-by-side comparison for each model
print("\n=== BEFORE vs AFTER COMPARISON ===\n")

for model_name in initial_responses:
    if model_name in reflected_responses:
        print(f"\n{'='*20} {model_name.upper()} {'='*20}\n")
        
        print("BEFORE REFLECTION:")
        print("-" * 50)
        print(initial_responses[model_name][:300] + "...")
        
        print("\nAFTER REFLECTION:")
        print("-" * 50)
        # Extract just the "Improved Response" section if it exists
        reflected = reflected_responses[model_name]
        if "## Improved Response" in reflected:
            improved_section = reflected.split("## Improved Response")[1].strip()
            print(improved_section[:400] + "...")
        else:
            print(reflected[:400] + "...")
        
        print("\n" + "="*70 + "\n")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Pattern Analysis</h2>
            <span style="color:#ff7800;">
            <b>Patterns used:</b><br/>
            1. <b>Multi-Model Comparison:</b> Comparing multiple models on the same task<br/>
            2. <b>Judge/Evaluator:</b> Using a model to evaluate performances<br/>
            3. <b>Reflection (NEW):</b> Self-critique and improvement of responses<br/><br/>
            <b>Possible experiments:</b><br/>
            - Iterate the Reflection Pattern multiple times<br/>
            - Add a "Debate Pattern" between models<br/>
            - Implement a "Consensus Pattern"
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial Applications</h2>
            <span style="color:#00bfff;">
            The <b>Reflection Pattern</b> is particularly valuable for:<br/>
            • Improving quality of complex analyses<br/>
            • Reducing bias in AI recommendations<br/>
            • Creating self-improving systems<br/>
            • Developing more robust AI for critical decisions<br/><br/>
            Use cases: Strategic consulting, risk analysis, ethical evaluation, medical diagnosis
            </span>
        </td>
    </tr>
</table>

## Additional Pattern Ideas for Future Implementation

In [ ]:
# 1. Chain of Thought Pattern
"""
Add a pattern that asks models to show their reasoning step by step:

def apply_chain_of_thought_pattern(client, question):
    prompt = f\"
    Question: {question}
    
    Please think through this step by step:
    Step 1: [Identify the key issues]
    Step 2: [Consider different perspectives]
    Step 3: [Evaluate potential consequences]
    Step 4: [Provide reasoned conclusion]
    \"
    return get_response(client, prompt)
"""

# 2. Iterative Refinement Pattern
"""
Create a loop that progressively improves the response over multiple iterations:

def iterative_refinement(client, question, iterations=3):
    response = get_initial_response(client, question)
    for i in range(iterations):
        critique_prompt = f\"Improve this response: {response}\"
        response = get_response(client, critique_prompt)
    return response
"""

# 3. Debate Pattern
"""
Make two models debate their respective responses:

def create_debate(client1, client2, question):
    response1 = get_response(client1, question)
    response2 = get_response(client2, question)
    
    debate_prompt1 = f\"Argue against this position: {response2}\"
    debate_prompt2 = f\"Argue against this position: {response1}\"
    
    counter1 = get_response(client1, debate_prompt1)
    counter2 = get_response(client2, debate_prompt2)
    
    return counter1, counter2
"""

# 4. Consensus Building Pattern
"""
Attempt to create a consensus response based on all individual responses:

def build_consensus(all_responses, question):
    consensus_prompt = f\"
    Original question: {question}
    
    Here are multiple expert responses:
    {all_responses}
    
    Create a consensus response that incorporates the best insights from all responses
    while resolving contradictions.
    \"
    return get_response(openai_client, consensus_prompt)
"""

print("Exercise completed! Analyze the results to see the impact of the Reflection Pattern.")